# Lab 8c (optional): The same agent in Google ADK
**Module 4.2, framework comparison.**  **Time:** 15 minutes.

Google's **Agent Development Kit (ADK)** is designed around Gemini but can use other models through LiteLLM.
It separates the **Agent** (model, instruction, tools) from the **Runner** (executes turns) and the
**Session** (conversation state). That split maps well onto platform architecture: state is a service.

In [ ]:
# SETUP: run this cell first. It works in Google Colab and in VS Code.
import os, sys, subprocess
REPO_URL = "https://github.com/cybertide-solutions/ai-platform-architect-labs.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/course"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/course"], check=True)
    os.chdir("/content/course")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # VS Code starts in notebooks/; the course files are one level up
sys.path.insert(0, os.getcwd())
import labkit as lk
lk.setup();

In [ ]:
import os
lk.pip_install("google-adk", "litellm", module="google.adk")

In [ ]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Use the provider labkit picked, through its OpenAI-compatible endpoint.
model = LiteLlm(model="openai/" + lk.model_name("big"), api_base=lk.CONFIG["base_url"],
                api_key=os.environ.get("GROQ_API_KEY") or os.environ.get("GEMINI_API_KEY") or "not-needed")

ORDERS = {"ORD-1001": {"status": "shipped", "amount_inr": 2499},
          "ORD-1002": {"status": "delivered", "amount_inr": 7999}}

def order_status(order_id: str) -> dict:
    """Get the status and amount of a customer order by its id, for example ORD-1001."""
    return ORDERS.get(order_id.upper(), {"error": "not found"})

support = Agent(name="support_agent", model=model, tools=[order_status],
                instruction="You are a Meridian Retail support agent. Check the order system before answering. Be brief.")

In [ ]:
session_service = InMemorySessionService()          # production: a database-backed session service
runner = Runner(agent=support, app_name="support", session_service=session_service)

async def ask(question: str, session_id: str | None = None):
    if session_id is None:
        s = await session_service.create_session(app_name="support", user_id="u1")
        session_id = s.id
    msg = types.Content(role="user", parts=[types.Part(text=question)])
    async for event in runner.run_async(user_id="u1", session_id=session_id, new_message=msg):
        for part in (event.content.parts if event.content else []):
            if part.function_call:
                print("TOOL CALL:  ", part.function_call.name, dict(part.function_call.args))
            if part.function_response:
                print("TOOL RESULT:", part.function_response.response)
            if part.text:
                print("AGENT:      ", part.text)
    return session_id

sid = await ask("Where is my order ORD-1001?")

In [ ]:
# Same session: the agent remembers the conversation
_ = await ask("And what was the amount on it?", sid)

**Using Gemini directly:** with a Gemini key you can skip LiteLLM: set `os.environ["GOOGLE_API_KEY"]` and use
`Agent(model="gemini-2.5-flash", ...)`.

**Local developer UI:** on a laptop, ADK includes a web UI for stepping through traces: `adk web` (see the ADK docs).

## Compare
| Concept | Lab 8 | CrewAI | ADK |
|---|---|---|---|
| instructions | system prompt | role, goal, backstory | `instruction` |
| loop | `for step` | `Crew.kickoff()` | `Runner.run_async()` |
| memory | the `msgs` list | crew memory | `SessionService` |